#**Demo proyecto final**

## **1. Conexión al drive para cargar modelo guardado**

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Carpeta del modelo (cambiar solo aquí)
RUTA_MODELO = './modelos/modelo_xlmr'
# Para la sustentación en Colab, con el modelo en Drive, usar esta línea en su lugar:
#RUTA_MODELO = '/content/drive/MyDrive/proyecto_final/preentrenados/modelo_xlmr'

UMBRAL = 0.65                     # elegido en validación

tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO).eval()

ETIQUETAS = {'negative': 'NEGATIVA', 'neutral': 'NEUTRAL', 'positive': 'POSITIVA'}
print('Modelo Roberta (XLM-R) cargado')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo Roberta (XLM-R) cargado


## **2. Función para la demo**

In [7]:
def clasificar(texto):
    print(f'Noticia: "{texto}"')

    # Manejo de error: texto vacío
    if not texto or not texto.strip():
        print('→ Texto vacío: no se puede clasificar. Va a REVISIÓN HUMANA.\n')
        return

    enc = tokenizer(texto, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        probs = torch.softmax(modelo(**enc).logits, dim=-1)[0]

    clase = modelo.config.id2label[int(probs.argmax())]
    confianza = float(probs.max())

    # Evidencia: probabilidad de cada clase
    for i, p in enumerate(probs):
        print(f'   {ETIQUETAS[modelo.config.id2label[i]]:9s} {float(p):.2f}')

    # Decisión con el umbral de abstención
    if confianza >= UMBRAL:
        print(f'→ {ETIQUETAS[clase]} (confianza {confianza:.2f}): el modelo DECIDE SOLO.\n')
    else:
        print(f'→ {ETIQUETAS[clase]} (confianza {confianza:.2f}): por debajo del umbral {UMBRAL}. Va a REVISIÓN HUMANA.\n')

#**3. Ejemplos para la presentación**

In [8]:
ejemplos = [
    'Las ventas de la compañía no subieron este trimestre',      # negación
    'Los ingresos crecieron, pero el beneficio cayó con fuerza', # contraste
    'El Ibex 35 sube un 2 % en la apertura',                     # positiva
    'Ibex35 +2% apertura',                                       # abreviatura: el modelo duda
    'La fotosíntesis depende de la energía solar',               # fuera de dominio
    '',                                                          # texto vacío
]
for texto in ejemplos:
    clasificar(texto)

Noticia: "Las ventas de la compañía no subieron este trimestre"
   NEGATIVA  0.96
   NEUTRAL   0.02
   POSITIVA  0.01
→ NEGATIVA (confianza 0.96): el modelo DECIDE SOLO.

Noticia: "Los ingresos crecieron, pero el beneficio cayó con fuerza"
   NEGATIVA  0.89
   NEUTRAL   0.07
   POSITIVA  0.04
→ NEGATIVA (confianza 0.89): el modelo DECIDE SOLO.

Noticia: "El Ibex 35 sube un 2 % en la apertura"
   NEGATIVA  0.03
   NEUTRAL   0.03
   POSITIVA  0.94
→ POSITIVA (confianza 0.94): el modelo DECIDE SOLO.

Noticia: "Ibex35 +2% apertura"
   NEGATIVA  0.01
   NEUTRAL   0.38
   POSITIVA  0.62
→ POSITIVA (confianza 0.62): por debajo del umbral 0.65. Va a REVISIÓN HUMANA.

Noticia: "La fotosíntesis depende de la energía solar"
   NEGATIVA  0.01
   NEUTRAL   0.97
   POSITIVA  0.02
→ NEUTRAL (confianza 0.97): el modelo DECIDE SOLO.

Noticia: ""
→ Texto vacío: no se puede clasificar. Va a REVISIÓN HUMANA.



# **4. Escribir frase**

In [9]:
clasificar(input('Escribe una noticia: '))

Escribe una noticia: las accciones de ecopetrol subiero drásticamente, pese a los esfuerzos de la junta por contrarrestar los efectos
Noticia: "las accciones de ecopetrol subiero drásticamente, pese a los esfuerzos de la junta por contrarrestar los efectos"
   NEGATIVA  0.86
   NEUTRAL   0.10
   POSITIVA  0.04
→ NEGATIVA (confianza 0.86): el modelo DECIDE SOLO.



#